# 06 · Incremental models

`fct_orders` uses the classic pattern: `is_incremental()` + high-water mark
+ `delete+insert`. `fct_events` uses **microbatch** (dbt 1.9+): one
independent batch per day. This notebook advances business time by one day
and watches both react.

Companion reading: [docs/11](../docs/11_incremental_deep_dive.md).

In [ ]:
# --- Standard setup (identical in every notebook) ---------------------------
import os, sys, json, subprocess
from pathlib import Path
from datetime import date, timedelta

from dotenv import load_dotenv

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
load_dotenv(REPO / ".env")

os.chdir(REPO / "jaffle_shop")          # dbt must run from the project dir
os.environ.setdefault("DBT_PROFILES_DIR", str(REPO / "jaffle_shop"))

from dbt.cli.main import dbtRunner

def run_dbt(args):
    """Run dbt programmatically (same engine as the CLI). Returns a
    dbtRunnerResult: .success says how it went, .result has per-node details.
    Crucially it never sys.exit()s -- perfect for notebooks."""
    print(f"$ dbt {' '.join(args)}")
    res = dbtRunner().invoke(args)
    print(f"-> success={res.success}")
    return res

def load_day(*flags):
    """Invoke the raw-data generator (plays the role of the EL tool)."""
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "generate_data.py"), *flags],
        check=True,
    )


In [ ]:
# --- Warehouse connection for %%sql cells (jupysql) and pandas --------------
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBT_USER', 'dbt')}:{os.getenv('DBT_PASSWORD', 'dbt')}"
    f"@{os.getenv('DBT_HOST', 'localhost')}:{os.getenv('DBT_PORT', '5432')}"
    f"/{os.getenv('DBT_DBNAME', 'jaffle_shop')}"
)

%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 25


## 1. Baseline: full-refresh `fct_orders`, note the row count

In [ ]:
res = run_dbt(["run", "--select", "fct_orders", "--full-refresh"])
assert res.success

baseline = pd.read_sql("select count(*) as n from dev_marts.fct_orders", engine)["n"][0]
next_day = int(pd.read_sql("select max(_batch_day) + 1 as d from raw.raw_orders", engine)["d"][0])
print(f"fct_orders baseline: {baseline} rows; about to load day {next_day}")

## 2. Business time advances: the EL tool syncs a new day

This inserts new orders AND mutates old rows (status changes bump
`updated_at`) -- exactly what a real operational source does.

In [ ]:
load_day("--day", str(next_day))

## 3. Incremental run -- watch the filter work

In [ ]:
res = run_dbt(["run", "--select", "fct_orders"])
assert res.success

after = pd.read_sql("select count(*) as n from dev_marts.fct_orders", engine)["n"][0]
print(f"{baseline} -> {after} rows (+{after - baseline})")

In [ ]:
# How many rows did the incremental run actually PROCESS? More than the net
# new count -- it also re-processed mutated old orders (delete+insert).
rr = json.loads(Path("target/run_results.json").read_text())
for r in rr["results"]:
    print(r["unique_id"].split(".")[-1], "->", r["adapter_response"].get("rows_affected"), "rows affected")

## 4. Proof in the compiled SQL: the `is_incremental()` branch

In [ ]:
compiled = Path("target/compiled/jaffle_shop/models/marts/fct_orders.sql").read_text()
start = compiled.find("where updated_at")
print("..." + compiled[start:start + 220] + "...")

On the full-refresh run this WHERE clause was absent (`is_incremental()`
was false). Now it filters staging to rows past the high-water mark
`max(updated_at)` of the already-built table.

## 5. Status changes really did propagate

In [ ]:
%%sql
select status, count(*) from dev_marts.fct_orders group by status order by 2 desc

## 6. Microbatch and the late-data lesson

`fct_events` batches by **event day**. Yesterday's batch and today's get
reprocessed each run (`lookback=1`) -- but the day we just loaded carries
*business* dates older than that, so a plain run does NOT pick it up.
This is the late-arriving-data problem, and microbatch's answer is an
explicit **backfill** of exactly the affected window:

In [ ]:
BASE = date(2026, 7, 1)   # business day 1 (see scripts/generate_data.py)
day_date = BASE + timedelta(days=next_day - 1)

before = pd.read_sql("select count(*) as n from dev_marts.fct_events", engine)["n"][0]

res = run_dbt(["run", "--select", "fct_events"])          # plain run: lookback window only
mid = pd.read_sql("select count(*) as n from dev_marts.fct_events", engine)["n"][0]

res = run_dbt(["run", "--select", "fct_events",           # backfill: exactly one day
               "--event-time-start", str(day_date),
               "--event-time-end", str(day_date + timedelta(days=1))])
assert res.success
after = pd.read_sql("select count(*) as n from dev_marts.fct_events", engine)["n"][0]

print(f"before={before}  after plain run={mid}  after backfill={after}")

The plain run added nothing; the targeted backfill loaded the new day. Each
batch is replaceable in isolation -- that is the whole point of microbatch.

## 7. Downstream of an incremental: refresh the aggregate

In [ ]:
res = run_dbt(["run", "--select", "agg_daily_revenue"])
assert res.success

In [ ]:
%%sql
select * from dev_marts.agg_daily_revenue order by order_date

## Rules of thumb

* Start with `view`/`table`; go incremental only when rebuilds actually hurt.
* `--full-refresh` must always be safe -- treat incremental state as a cache.
* Pick the strategy by how your data changes: append-only -> `append`,
  mutable keys -> `delete+insert`/`merge`, event streams -> `microbatch`.